In [ ]:
import glob
import os
import re
import requests
import numpy as np
import pandas as pd

### Create full dataset

Create dataset by concatenating all scraping datasets and convert price from `грн/міс` to `$/міс`.

In [ ]:
file_path = '../data'

files = glob.glob(os.path.join(file_path, 'scrapped/*.csv'))

def convert_to_usd(row):
    if row['currency'] == 'грн/міс':
        return int(np.round(row['price'] / usd_rate, -1))
    return row['price']

def df_convert_price(df: pd.DataFrame) -> pd. DataFrame:

    df.dropna(subset=['price'], inplace=True)
    df['price'] = df['price'].str.replace(' ', '').astype('int32')

    df['price'] = df.apply(convert_to_usd, axis=1)

    df.drop(columns=['currency'], inplace=True)

all_df = []

for file in files:
    date = re.search('\d{8}', file)
    scrap_date = date.group()

    # get usd_rate for current scrap_date
    resp = requests.get(f'https://bank.gov.ua/NBUStatService/v1/statdirectory/exchangenew?json&valcode=USD&date={scrap_date}')
    for obj in resp.json():
        usd_rate = obj['rate']

    df = pd.read_csv(file_path + '/' + file)
    df_convert_price(df)
    all_df.append(df)

data = pd.concat(all_df, axis=0, ignore_index=True)

<>:22: SyntaxWarning: invalid escape sequence '\d'
<>:22: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Scar\AppData\Local\Temp\ipykernel_12872\3910526527.py:22: SyntaxWarning: invalid escape sequence '\d'
  date = re.search('\d{8}', file)


In [6]:
data.shape

(21831, 15)

In [7]:
data.sample(3)

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,residential,neighborhood,description,detail,attributes
7899,11396853,470,"Саксаганського вул., 70/16","30.50267982,50.43920517","Київ,Голосіївський р-н",Університет,2 кімнати,45 / 30.7 / 7.6 м²,поверх 4 з 6,"['Українська цегла', 'Суміжна', 'Перша здача']",NaN,"['Паньківщина', 'Ботанічний сад ім. акад. О. В...",Здам довгостроково для сімейної пари 2-кімнатн...,"Будинок - Українська цегла, в квартирі 2 кімна...","['Лічильники', 'Кондиціонер', 'Мікрохвильовка'..."
20222,11393702,2500,"Назарівська вул. (Вєтрова), 11","30.49997902,50.44282913","Київ,Шевченківський р-н",Університет,4 кімнати,175 / 90 / 15 м²,поверх 12 з 25,"['Бетонно монолітний', 'Багаторівнева', 'Дизай...",NaN,"['Ботанічний сад ім. акад. О. В. Фоміна', 'Пан...",Актуально! Можливі перегляди!\n\nПропонуємо в ...,"Будинок - Бетонно монолітний, в квартирі 4 кім...","['Душова кабіна', 'Кондиціонер', 'Пральна маши..."
10879,11407069,220,"Мілютенка вул., 18","30.63216019,50.47116852","Київ,Деснянський р-н",Лісова,2 кімнати,52 / 30 / 8 м²,поверх 7 з 9,"['Роздільне', 'Хороший стан']",NaN,"['Лісовий масив', 'Парк Кіото']",Метро Лісова хвилин 15-20 пішки. Лісовий масив...,В квартирі 2 кімнати. Планування кімнат Розділ...,[]


In [ ]:
data.to_csv('../data/row/apartments_row.csv', index=False)

Drop duplicated rows.

In [9]:
data.duplicated().sum()

8026

In [10]:
data.drop_duplicates(keep='last', inplace=True, ignore_index=True)

In [11]:
data.shape

(13805, 15)

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13805 entries, 0 to 13804
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            13805 non-null  int64 
 1   price         13805 non-null  int64 
 2   address       13805 non-null  object
 3   coordinates   13642 non-null  object
 4   region        13805 non-null  object
 5   subway        11256 non-null  object
 6   rooms         13805 non-null  object
 7   footage       13805 non-null  object
 8   floor         13805 non-null  object
 9   features      13805 non-null  object
 10  residential   6556 non-null   object
 11  neighborhood  13805 non-null  object
 12  description   13805 non-null  object
 13  detail        13804 non-null  object
 14  attributes    13805 non-null  object
dtypes: int64(2), object(13)
memory usage: 1.6+ MB


### Drop all possible duplicates

In [ ]:
data.shape

(13805, 15)

In [15]:
data[data.duplicated(subset=['id'], keep=False)].sort_values(by=['id'])

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,residential,neighborhood,description,detail,attributes
6854,8444376,790,"Сікорського Ігоря вул. (Танкова), 4 г",NaN,"Київ,Шевченківський р-н",Сирець,3 кімнати,110 / 70 / 12 м²,поверх 11 з 25,"['Українська цегла', 'Роздільне', 'Євроремонт']",NaN,[],БЕЗ КОМИССИИ.сдается эксклюзивная 3-ком. кв. 1...,"Будинок - Українська цегла, в квартирі 3 кімна...","['Посудомийна машина', 'Джакузі', 'Лічильники'..."
4961,8444376,800,"Сікорського Ігоря вул. (Танкова), 4 г",NaN,"Київ,Шевченківський р-н",Сирець,3 кімнати,110 / 70 / 12 м²,поверх 11 з 25,"['Українська цегла', 'Роздільне', 'Євроремонт']",NaN,[],БЕЗ КОМИССИИ.сдается эксклюзивная 3-ком. кв. 1...,"Будинок - Українська цегла, в квартирі 3 кімна...","['Посудомийна машина', 'Джакузі', 'Лічильники'..."
11931,8852403,900,"Сікорського Ігоря вул. (Танкова), 4Г","30.42969894,50.46660614","Київ,Шевченківський р-н",Сирець,2 кімнати,80 / 60 / 20 м²,поверх 12 з 24,"['Суміжно-роздільна', 'Євроремонт']",ЖК Зелений острів-2,['Парк Нивки'],БЕЗ КОМИССИИ.Впервые сдается эксклюзивная \n2-...,В квартирі 2 кімнати. Планування кімнат Суміжн...,"['Посудомийна машина', 'Душова кабіна', 'Джаку..."
2374,8852403,900,"Сікорського Ігоря вул. (Танкова), 4Г","30.42969894,50.46660614","Київ,Шевченківський р-н",Сирець,2 кімнати,80 / 60 / 20 м²,поверх 12 з 24,"['Суміжно-роздільна', 'Євроремонт']",ЖК Зелений острів-2,['Парк Нивки'],БЕЗ КОМИССИИ.Впервые сдается эксклюзивная \n2-...,В квартирі 2 кімнати. Планування кімнат Суміжн...,[]
2122,9489042,2399,"Ярославів Вал вул., 28","30.50898933,50.45216751","Київ,Шевченківський р-н",Золоті Ворота,4 кімнати,130 / 70 / 25 м²,поверх 5 з 5,"['Дореволюційний', 'Роздільне', 'Євроремонт']",NaN,"['Львівська площа', 'Старий Київ', 'Пейзажна а...",БЕЗ КОМИССИИ !! \nВладелец. 4-х комнатная квар...,"Будинок - Дореволюційний, в квартирі 4 кімнати...","['Посудомийна машина', 'Душова кабіна', 'Сейф'..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11814,11427332,950,Володимира Івасюка просп. (Героїв Сталінграда)...,"30.51220512,50.50650787","Київ,Оболонський р-н",Мінська,3 кімнати,143 / 70.5 / 17 м²,поверх 21 з 27,"['Українська цегла', 'Суміжно-роздільна', 'Чуд...",NaN,"['Оболонь', 'Оболонська набережна']",пропонуємо 3 кімнатну квартиру на оренду на Об...,"Будинок - Українська цегла, в квартирі 3 кімна...",[]
11746,11427357,840,"Героїв полку ""Азов"" вул. (Малиновського маршал...","30.48916245,50.50515366","Київ,Оболонський р-н",Оболонь,4 кімнати,125 / 96 / 15 м²,поверх 12 з 24,['Євроремонт'],ЖК Гранд,['Оболонь'],ДЗВОНІТЬ ЗА ВКАЗАНИМ НОМЕРОМ ТЕЛЕФОНУ. ЯКЩО НЕ...,В квартирі 4 кімнати. Загальний стан квартири ...,[]
8887,11427357,860,"Героїв полку ""Азов"" вул. (Малиновського маршал...","30.48916245,50.50515366","Київ,Оболонський р-н",Оболонь,4 кімнати,125 / 96 / 15 м²,поверх 12 з 24,['Євроремонт'],ЖК Гранд,['Оболонь'],ДЗВОНІТЬ ЗА ВКАЗАНИМ НОМЕРОМ ТЕЛЕФОНУ. ЯКЩО НЕ...,В квартирі 4 кімнати. Загальний стан квартири ...,[]
11549,11427452,800,Володимира Івасюка просп. (Героїв Сталінграда)...,"30.5214653,50.49625778","Київ,Оболонський р-н",Оболонь,2 кімнати,89 / 50 / 22 м²,поверх 5 з 9,"['Бетонно монолітний', 'Роздільне', 'Чудовий с...",NaN,"['Парк Наталка', 'Оболонська набережна']",Прекрасна 2-кімнатна квартира з роздільним пла...,"Будинок - Бетонно монолітний, в квартирі 2 кім...",[]


It looks like the same apartment ads in one case contain information in `attributes` or `neighborhood` and in other they don't. 
Let's try to fill this empty values.

Let's replace empty values `[]` with NaN.

In [16]:
data.replace('[]', np.nan, inplace=True)

In [17]:
data.isna().sum()

id                 0
price              0
address            0
coordinates      163
region             0
subway          2549
rooms              0
footage            0
floor              0
features           0
residential     7249
neighborhood    1354
description        0
detail             1
attributes      5575
dtype: int64

Replace NaN value with value from another ads with the same `id`.

In [18]:
data.index.name = 'index'

In [19]:
data_dupl = (data[data.duplicated(subset=['id'], keep=False)].sort_values(by=['id', 'index'])
             [['id', 'coordinates', 'subway', 'residential', 'neighborhood', 'attributes']])
data_dupl

,id,coordinates,subway,residential,neighborhood,attributes
index,,,,,,
4961,8444376,NaN,Сирець,NaN,NaN,"['Посудомийна машина', 'Джакузі', 'Лічильники'..."
6854,8444376,NaN,Сирець,NaN,NaN,"['Посудомийна машина', 'Джакузі', 'Лічильники'..."
2374,8852403,"30.42969894,50.46660614",Сирець,ЖК Зелений острів-2,['Парк Нивки'],NaN
11931,8852403,"30.42969894,50.46660614",Сирець,ЖК Зелений острів-2,['Парк Нивки'],"['Посудомийна машина', 'Душова кабіна', 'Джаку..."
2122,9489042,"30.50898933,50.45216751",Золоті Ворота,NaN,"['Львівська площа', 'Старий Київ', 'Пейзажна а...","['Посудомийна машина', 'Душова кабіна', 'Сейф'..."
...,...,...,...,...,...,...
11814,11427332,"30.51220512,50.50650787",Мінська,NaN,"['Оболонь', 'Оболонська набережна']",NaN
8887,11427357,"30.48916245,50.50515366",Оболонь,ЖК Гранд,['Оболонь'],NaN
11746,11427357,"30.48916245,50.50515366",Оболонь,ЖК Гранд,['Оболонь'],NaN


In [20]:
features = ['coordinates', 'subway', 'residential', 'neighborhood', 'attributes']

for feat in features:
    print(f'number of missing values in column `{feat}`')
    data_dupl[feat] = data_dupl[feat].fillna(
        data_dupl['id'].map(data_dupl
                            .dropna(subset=[feat])
                            .drop_duplicates(subset=['id'])
                            .set_index('id')
                            [feat]
                            )
    )
    print('before:', data[feat].isna().sum())
    data[feat] = data[feat].fillna(data_dupl[feat])
    print('after:', data[feat].isna().sum())
    print('---------------', '\n')

number of missing values in column `coordinates`
before: 163
after: 163
--------------- 

number of missing values in column `subway`
before: 2549
after: 2549
--------------- 

number of missing values in column `residential`
before: 7249
after: 7242
--------------- 

number of missing values in column `neighborhood`
before: 1354
after: 1327
--------------- 

number of missing values in column `attributes`
before: 5575
after: 5425
--------------- 



In [21]:
(data[data.duplicated(subset=data.columns.difference(['id', 'price']), keep=False)]
    .sort_values(by=['address', 'footage', 'index']))

,id,price,address,coordinates,region,subway,rooms,footage,floor,features,residential,neighborhood,description,detail,attributes
index,,,,,,,,,,,,,,,
7572,11338552,900,"Євгена Коновальця вул. (Щорса), 32б","30.5324955,50.42583847","Київ,Печерський р-н",Печерська,3 кімнати,150 / 100 / 15 м²,поверх 14 з 24,"['Українська цегла', 'Роздільне', 'Євроремонт']",NaN,['КНУКіМ'],Пропонується в оренду 3-кімнатна квартира на П...,"Будинок - Українська цегла, в квартирі 3 кімна...","['Посудомийна машина', 'Душова кабіна', 'Лічил..."
9678,11338552,800,"Євгена Коновальця вул. (Щорса), 32б","30.5324955,50.42583847","Київ,Печерський р-н",Печерська,3 кімнати,150 / 100 / 15 м²,поверх 14 з 24,"['Українська цегла', 'Роздільне', 'Євроремонт']",NaN,['КНУКіМ'],Пропонується в оренду 3-кімнатна квартира на П...,"Будинок - Українська цегла, в квартирі 3 кімна...","['Посудомийна машина', 'Душова кабіна', 'Лічил..."
5382,11392662,950,"Євгена Коновальця вул. (Щорса), 32б","30.5324955,50.42583847","Київ,Печерський р-н",Печерська,3 кімнати,150 / 85 / 15 м²,поверх 14 з 21,"['Українська цегла', 'Роздільне', 'Євроремонт']",NaN,['КНУКіМ'],Видова 3к квартира 150м Митець Коновальця 32Б ...,"Будинок - Українська цегла, в квартирі 3 кімна...",NaN
12746,11392662,940,"Євгена Коновальця вул. (Щорса), 32б","30.5324955,50.42583847","Київ,Печерський р-н",Печерська,3 кімнати,150 / 85 / 15 м²,поверх 14 з 21,"['Українська цегла', 'Роздільне', 'Євроремонт']",NaN,['КНУКіМ'],Видова 3к квартира 150м Митець Коновальця 32Б ...,"Будинок - Українська цегла, в квартирі 3 кімна...",NaN
1975,11140005,2300,"Євгена Коновальця вул. (Щорса), 34а","30.53315544,50.42731857","Київ,Печерський р-н",Печерська,4 кімнати,183 / 105 / 25 м²,поверх 20 з 25,"['Українська цегла', 'Роздільне', 'Дизайнерськ...",ЖК Аристократ,"['КНУКіМ', 'Військовий госпіталь', 'Пологовий ...","Пропонується в оренду шикарна, видова, 4х кімн...","Будинок - Українська цегла, в квартирі 4 кімна...","['Посудомийна машина', 'Душова кабіна', 'Джаку..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6491,11302005,1100,"проспект Володимира Івасюка, 12 Е","30.51338196,50.50621796","Київ,Оболонський р-н",Мінська,2 кімнати,98 / 34 / 38 м²,поверх 10 з 26,"['Бетонно монолітний', 'Роздільне', 'Дизайнерс...",NaN,"['Оболонська набережна', 'Оболонь']",Пропонується унікальна 2-х кімнатна квартира в...,"Будинок - Бетонно монолітний, в квартирі 2 кім...","['Камін', 'Посудомийна машина', 'Джакузі', 'Лі..."
4762,11393710,440,"проспект Любомира Гузара, 15","30.40484047,50.43284607","Київ,Солом'янський р-н",NaN,1 кімната,30 / 22 / 8 м²,поверх 13 з 17,"['Бетонно монолітний', 'Роздільне', 'Дизайнерс...",ЖК Сучасний квартал,['Медмістечко'],"Чудова, затишна квартира в новому комплексі ""С...","Будинок - Бетонно монолітний, в квартирі 1 кім...","['Душова кабіна', 'Лічильники', 'Кондиціонер',..."
6583,11393710,430,"проспект Любомира Гузара, 15","30.40484047,50.43284607","Київ,Солом'янський р-н",NaN,1 кімната,30 / 22 / 8 м²,поверх 13 з 17,"['Бетонно монолітний', 'Роздільне', 'Дизайнерс...",ЖК Сучасний квартал,['Медмістечко'],"Чудова, затишна квартира в новому комплексі ""С...","Будинок - Бетонно монолітний, в квартирі 1 кім...","['Душова кабіна', 'Лічильники', 'Кондиціонер',..."


In [22]:
data_dupl_two = (data[data.duplicated(subset=df.columns.difference(['id', 'price']), keep=False)]
                 .sort_values(by=['address', 'footage', 'index'])
                 [['address', 'coordinates', 'subway', 'residential', 'neighborhood', 'attributes']])
data_dupl_two.shape

(2948, 6)

In [23]:
features = ['coordinates', 'subway', 'residential', 'neighborhood', 'attributes']

for feat in features:
    print(f'number of missing values in column `{feat}`')
    data_dupl_two[feat] = data_dupl_two[feat].fillna(
        data_dupl_two['address'].map(data_dupl_two
                            .dropna(subset=[feat])
                            .drop_duplicates(subset=['address'])
                            .set_index('address')
                            [feat]
                            )
    )
    print('before:', data[feat].isna().sum())
    data[feat] = data[feat].fillna(data_dupl_two[feat])
    print('after:', data[feat].isna().sum())
    print('---------------', '\n')

number of missing values in column `coordinates`
before: 163
after: 153
--------------- 

number of missing values in column `subway`
before: 2549
after: 2541
--------------- 

number of missing values in column `residential`
before: 7242
after: 7163
--------------- 

number of missing values in column `neighborhood`
before: 1327
after: 1302
--------------- 

number of missing values in column `attributes`
before: 5425
after: 5169
--------------- 



Drop rows, where the same information except only `id` and `price`

In [24]:
data.duplicated(subset=data.columns.difference(['id', 'price']), keep=False).sum()

2948

In [25]:
data.drop_duplicates(subset=data.columns.difference(['id', 'price']),
                     keep='last',
                     inplace=True,
                     ignore_index=True)
data.shape

(12224, 15)

In [26]:
data.duplicated(subset=['id']).sum()

788

In [ ]:
data.drop_duplicates(subset=['id'],
                     keep='last',
                     inplace=True,
                     ignore_index=True)
data.shape

(11436, 15)

In [28]:
data.shape

(11436, 15)

In [ ]:
data.to_csv('../data/row/apartments_without_duplicates.csv', index=False)